## 文档加载器

In [5]:
%pip install unstructured
%pip install markdown

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 1.创建文档加载器，并指定路径
document_load = UnstructuredMarkdownLoader(file_path="LangChain框架入门09：什么是RAG？.md",mode="elements")

# 2.加载文档
documents = document_load.load()

# 3.打印文档内容
print(f"文档数量：{len(documents)}")
for document in documents:
    print(f"文档内容：{document.page_content}")
    print(f"文档元数据：{document.metadata}")

文档数量：6
文档内容：什么是 RAG
文档元数据：{'source': 'LangChain框架入门09：什么是RAG？.md', 'category_depth': 0, 'languages': ['zho'], 'filename': 'LangChain框架入门09：什么是RAG？.md', 'filetype': 'text/markdown', 'last_modified': '2026-05-14T14:43:43', 'category': 'Title', 'element_id': 'c2c34c66e85aa79bc53bb20fcf7a59c2'}
文档内容：RAG 全称 Retrieval-Augmented Generation，中文译为检索增强生成，是当下大模型应用开发中最核心、最常用的技术架构。
文档元数据：{'source': 'LangChain框架入门09：什么是RAG？.md', 'emphasized_text_contents': ['Retrieval-Augmented Generation', '检索增强生成'], 'emphasized_text_tags': ['b', 'b'], 'languages': ['zho'], 'filename': 'LangChain框架入门09：什么是RAG？.md', 'filetype': 'text/markdown', 'last_modified': '2026-05-14T14:43:43', 'parent_id': 'c2c34c66e85aa79bc53bb20fcf7a59c2', 'category': 'UncategorizedText', 'element_id': '44130eb8ead2bf589b075f8fe6e60e7e'}
文档内容：传统大模型只依赖自身训练的静态知识，存在三大痛点：知识有截止时间、无法使用私有本地文档、容易产生一本正经的错误回答（幻觉）。
文档元数据：{'source': 'LangChain框架入门09：什么是RAG？.md', 'languages': ['zho'], 'filename': 'LangChain框架入门09：什么是RAG？.md', 'filetype': 'text/markdown',

### 自定义文档加载器
在实际开发中，基于文件的不同类型和不同格式，有时通过这些LangChain提供的文档加载器很难满足业务需求，例如需要根据特定规则提取文本片段，这时就需要开发自定义加载器，只需要定义一个自定义文档加载器类，并继承前面提到的BaseLoader类。

假设有如下需求，对 faq.txt文件进行文档加载，内容如下，要求将问题和答案加载成一个文档，并添加文件创建日期元数据。

Q：在线支付取消订单后钱怎么返还？

订单取消后，款项会在一个工作日内，直接返还到您的美团账户余额。

Q：怎么查看退款是否成功？

退款会在一个工作日之内到美团账户余额，可在“账号管理——我的账号”中查看是否到账。

Q：美团账户里的余额怎么提现？

余额可到美团网（meituan.com）——“我的美团→美团余额”里提取到您的银行卡或者支付宝账号，另外，余额也可直接用于支付外卖订单（限支持在线支付的商家）。

Q：余额提现到账时间是多久？

1-7个工作日内可退回您的支付账户。由于银行处理可能有延迟，具体以账户的到账时间为准。

Q：申请退款后，商家拒绝了怎么办？

申请退款后，如果商家拒绝，此时回到订单页面点击“退款申诉”，美团客服介入处理。

Q：怎么取消退款呢？

请在订单页点击“不退款了”，商家还会正常送餐的。

Q：前面下了一个在线支付的单子，由于未付款，订单自动取消了，这单会计算我的参与活动次数吗？

不会。如果是未支付的在线支付订单，可以先将订单取消（如果不取消需要15分钟后系统自动取消），订单无效后，此时您再下单仍会享受活动的优惠。

Q：为什么我用微信订餐，却无法使用在线支付？

目前只有网页版和美团外卖手机App(非美团手机客户端)订餐，才能使用在线支付，请更换到网页版和美团外卖手机App下单。

Q：如何进行付款？

美团外卖现在支持货到付款与在线支付，其中微信版与手机触屏版暂不支持在线支付。

In [9]:
import os
from datetime import datetime
from langchain_core.documents import Document
from langchain_community.document_loaders.base import BaseLoader

class SimpleQALoader(BaseLoader):
    """
    简单的问答文件加载器
    
    该加载器用于从文本文件中加载问答对，文件格式要求每两行为一组，
    第一行为问题(Q)，第二行为答案(A)
    
    Args:
        file_path (str): 问答文件的路径
        time_fmt (str): 时间格式字符串，默认为 "%Y-%m-%d %H:%M:%S"
    """

    def __init__(self, file_path: str, time_fmt: str = "%Y-%m-%d %H:%M:%S"):
        self.file_path = file_path
        self.time_fmt = time_fmt

    def load(self):
        """
        加载并解析问答文件
        
        读取文件中的问答对，每两行构成一个问答文档，第一行为问题，第二行为答案。
        每个文档包含问题和答案的组合内容，以及文件的元数据信息。
        
        Returns:
            list[Document]: 包含问答内容的文档列表，每个文档包含page_content和metadata
        """
        with open(self.file_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        docs = []
        created_ts = os.path.getctime(self.file_path)
        created_at = datetime.fromtimestamp(created_ts).strftime(self.time_fmt)

        # 每两行构成一个 Q/A
        for i in range(0, len(lines), 2):
            q = lines[i].lstrip("Q：:").strip()
            a = lines[i+1].lstrip("A：:").strip()
            page_content = f"Q: {q}\nA: {a}"

            doc = Document(
                page_content=page_content,
                metadata={
                    "source": self.file_path,
                    "created_at": created_at,
                }
            )
            docs.append(doc)

        return docs


# 使用示例
if __name__ == "__main__":
    loader = SimpleQALoader("faq.txt")
    docs = loader.load()
    print(f"共解析到 {len(docs)} 个文档")
    for i, d in enumerate(docs, 1):
        print(f"\n--- 文档 {i} ---")
        print(d.page_content)
        print("元数据：", d.metadata)

共解析到 9 个文档

--- 文档 1 ---
Q: 在线支付取消订单后钱怎么返还？
A: 订单取消后，款项会在一个工作日内，直接返还到您的美团账户余额。
元数据： {'source': 'faq.txt', 'created_at': '2026-05-14 14:51:23'}

--- 文档 2 ---
Q: 怎么查看退款是否成功？
A: 退款会在一个工作日之内到美团账户余额，可在“账号管理——我的账号”中查看是否到账。
元数据： {'source': 'faq.txt', 'created_at': '2026-05-14 14:51:23'}

--- 文档 3 ---
Q: 美团账户里的余额怎么提现？
A: 余额可到美团网（meituan.com）——“我的美团→美团余额”里提取到您的银行卡或者支付宝账号，另外，余额也可直接用于支付外卖订单（限支持在线支付的商家）。
元数据： {'source': 'faq.txt', 'created_at': '2026-05-14 14:51:23'}

--- 文档 4 ---
Q: 余额提现到账时间是多久？
A: 1-7个工作日内可退回您的支付账户。由于银行处理可能有延迟，具体以账户的到账时间为准。
元数据： {'source': 'faq.txt', 'created_at': '2026-05-14 14:51:23'}

--- 文档 5 ---
Q: 申请退款后，商家拒绝了怎么办？
A: 申请退款后，如果商家拒绝，此时回到订单页面点击“退款申诉”，美团客服介入处理。
元数据： {'source': 'faq.txt', 'created_at': '2026-05-14 14:51:23'}

--- 文档 6 ---
Q: 怎么取消退款呢？
A: 请在订单页点击“不退款了”，商家还会正常送餐的。
元数据： {'source': 'faq.txt', 'created_at': '2026-05-14 14:51:23'}

--- 文档 7 ---
Q: 前面下了一个在线支付的单子，由于未付款，订单自动取消了，这单会计算我的参与活动次数吗？
A: 不会。如果是未支付的在线支付订单，可以先将订单取消（如果不取消需要15分钟后系统自动取消），订单无效后，此时您再下单仍会享受活动的优惠。
元数据： {'s

## 文本分割器

分割文本

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1.分割文本内容
content = (
    "大模型RAG（检索增强生成）是一种结合生成模型与外部知识检索的技术，通过从大规模文档或数据库中检索相关信息，辅助生成模型以提升回答的准确性和相关性。其核心流程包括用户输入查询、系统检索相关知识、生成模型基于检索结果生成内容，并输出最终答案。RAG的优势在于能够弥补生成模型的知识盲区，提供更准确、实时和可解释的输出，广泛应用于问答系统、内容生成、客服、教育和企业领域。然而，其也面临依赖高质量知识库、可能的响应延迟、较高的维护成本以及数据隐私等挑战。")
# 2.定义递归文本分割器
# 使用RecursiveCharacterTextSplitter创建文本分割器，设置块大小为100，重叠长度为30
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=30, length_function=len)

# 3.分割文本
# 将原始文本内容分割成多个文本块
splitter_texts = text_splitter.split_text(content)

# 4.转换为文档对象
# 将分割后的文本块转换为文档对象列表
splitter_documents = text_splitter.create_documents(splitter_texts)
print(f"原始文本大小：{len(content)}")
print(f"分割文档数量：{len(splitter_documents)}")
for splitter_document in splitter_documents:
    print(f"文档片段大小：{len(splitter_document.page_content)},文档内容：{splitter_document.page_content}")


原始文本大小：225
分割文档数量：3
文档片段大小：100,文档内容：大模型RAG（检索增强生成）是一种结合生成模型与外部知识检索的技术，通过从大规模文档或数据库中检索相关信息，辅助生成模型以提升回答的准确性和相关性。其核心流程包括用户输入查询、系统检索相关知识、生成模
文档片段大小：100,文档内容：相关性。其核心流程包括用户输入查询、系统检索相关知识、生成模型基于检索结果生成内容，并输出最终答案。RAG的优势在于能够弥补生成模型的知识盲区，提供更准确、实时和可解释的输出，广泛应用于问答系统、内容
文档片段大小：85,文档内容：区，提供更准确、实时和可解释的输出，广泛应用于问答系统、内容生成、客服、教育和企业领域。然而，其也面临依赖高质量知识库、可能的响应延迟、较高的维护成本以及数据隐私等挑战。


分割文档对象

In [12]:
!pip install langchain_unstructured

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_unstructured import UnstructuredLoader

# 1.创建文档加载器，进行文档加载
loader = UnstructuredLoader("rag.txt")
documents = loader.load()

# 2.定义递归文本分割器
# 创建RecursiveCharacterTextSplitter实例，用于将文档分割成指定大小的文本块
# chunk_size: 每个文本块的最大字符数为100
# chunk_overlap: 相邻文本块之间的重叠字符数为30
# length_function: 使用len函数计算文本长度
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=30, length_function=len)

# 3.分割文本
# 使用文本分割器将加载的文档分割成多个较小的文档片段
splitter_documents = text_splitter.split_documents(documents)

# 输出分割后的文档信息
print(f"分割文档数量：{len(splitter_documents)}")
for splitter_document in splitter_documents:
    print(f"文档片段：{splitter_document.page_content}")
    print(f"文档片段大小：{len(splitter_document.page_content)}, 文档元数据：{splitter_document.metadata}")


分割文档数量：18
文档片段：什么是 RAG
文档片段大小：7, 文档元数据：{'source': 'rag.txt', 'last_modified': '2026-05-14T14:59:18', 'languages': ['zho'], 'filename': 'rag.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '900d2265de612a5626f047ce9ac28c7d'}
文档片段：RAG 全称 *
文档片段大小：8, 文档元数据：{'source': 'rag.txt', 'last_modified': '2026-05-14T14:59:18', 'languages': ['zho'], 'filename': 'rag.txt', 'filetype': 'text/plain', 'parent_id': '900d2265de612a5626f047ce9ac28c7d', 'category': 'UncategorizedText', 'element_id': '7761634d6b8d997ba4610a0fe78b779f'}
文档片段：Retrieval
文档片段大小：9, 文档元数据：{'source': 'rag.txt', 'last_modified': '2026-05-14T14:59:18', 'languages': ['zho'], 'filename': 'rag.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '110ac5ca7730ceb90d08ba10b29f45b0'}
文档片段：Augmented Generation*
文档片段大小：21, 文档元数据：{'source': 'rag.txt', 'last_modified': '2026-05-14T14:59:18', 'languages': ['zho'], 'filename': 'rag.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '2fe0c5833b25f4

按标题分割Markdown文件

In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# 1.文档加载
# 创建文本加载器并加载Markdown文档
loader = TextLoader(file_path="LangChain框架入门09：什么是RAG？.md")
documents = loader.load()
document_text = documents[0].page_content

# 2.定义文本分割器，设置指定要分割的标题
# 配置Markdown标题分割规则，指定不同级别的标题标记及其对应的元数据标签
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2")
]
headers_text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 3.按标题分割文档
# 使用标题分割器将文档按Markdown标题结构进行分割
headers_splitter_documents = headers_text_splitter.split_text(document_text)

print(f"按标题分割文档数量：{len(headers_splitter_documents)}")
for splitter_document in headers_splitter_documents:
    print(f"按标题分割文档片段大小：{len(splitter_document.page_content)}, 文档元数据：{splitter_document.metadata}")

# 4.定义递归文本分割器
# 创建递归字符分割器，用于进一步细分过大的文档片段
# chunk_size: 每个文本块的目标大小为100个字符
# chunk_overlap: 相邻文本块之间的重叠字符数为30
# length_function: 使用len函数计算文本长度
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                               chunk_overlap=30,
                                               length_function=len
                                              )

# 5.递归分割文本
# 对已按标题分割的文档片段进行二次递归分割，确保每个片段不超过指定大小
recursive_documents = text_splitter.split_documents(headers_splitter_documents)
print(f"第二次递归文本分割文档数量：{len(recursive_documents)}")
for recursive_document in recursive_documents:
    print(
        f"第二次递归文本分割文档片段大小：{len(recursive_document.page_content)}, 文档元数据：{recursive_document.metadata}")


按标题分割文档数量：1
按标题分割文档片段大小：526, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档数量：8
第二次递归文本分割文档片段大小：81, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：62, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：94, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：78, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：58, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：53, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：24, 文档元数据：{'Header 1': '什么是 RAG'}
第二次递归文本分割文档片段大小：82, 文档元数据：{'Header 1': '什么是 RAG'}
